**Exploring the socioeconomic variables**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pycountry

: 

In [ ]:
df = pd.read_csv('../compact.csv')

Which countries are represented?

In [ ]:
df['country'].unique()

We see that some of the "countries" are actually groups of countries such as "Africa", "European Union", and "World". The data for these locations is likely aggregated from all countries belonging to those categories. We may want to drop these rows to focus on individual countries.

In [ ]:
non_countries = ['Africa', 'Asia', 'Asia excl. China', 'Europe', 'European Union (27)', 'High-income countries', 'Low-income countries', 'Lower-middle-income countries', 'North America', 'Oceania', 'South America', 'Summer Olympics 2020', 'Upper-middle-income countries', 'Winter Olympics 2022', 'World', 'World excl. China', 'World excl. China and South Korea', 'World excl. China, South Korea, Japan and Singapore']

Some socioeconomic and demographic covariates that seem of interest and have reliable measurements:

In [ ]:
cols = ['population_density', 'median_age', 'gdp_per_capita', 'extreme_poverty', 'handwashing_facilities', 'hospital_beds_per_thousand', 'life_expectancy']

How often is the measurement of each covariate updated for each country? In other words, how many unique values of each covariate are recorded for each country?

In [ ]:
df_counts = df.groupby('country')[cols].nunique().reset_index()
#Exclude non-country countries
df_counts = df_counts[~df_counts['country'].isin(non_countries)]
df_counts

In [ ]:
for col in cols:
    print(col, df_counts[col].unique())

We see that each covariate has at most one measurement recorded for each country. For some variables and countries, there is no measurement, only NaN.

For how many countries is each covariate recorded?

In [ ]:
df_counts[cols].sum()

We see that many countries do not have handwashing facilities recorded.

For how many countries in the top 10 by GDP is each variable recorded?

In [ ]:
#The top 10 countries in the world by GDP (according to Wikipedia):
top_10_gdp = ['United States', 'China', 'Germany', 'Japan', 'United Kingdom', 'India', 'France', 'Italy', 'Russia', 'Brazil']
df_top10 = df_counts[df_counts['country'].isin(top_10_gdp)]
df_top10[cols].sum()

In [ ]:
df_top10[['country', 'handwashing_facilities']]

We see that only India and China have measurements of handwashing facilities. We may want to exclude this covariate so we aren't missing measurements for eight of the top ten countries by GDP.

If we only keep countries that have measurements of every covariate except handwashing facilities, how many countries are left?

In [ ]:
df_counts_new = df_counts.drop(columns = ['handwashing_facilities'])
df_counts_new = df_counts_new[(df_counts_new != 0).all(axis=1)]
countries_kept = list(df_counts_new['country'])

In [ ]:
print(countries_kept)

In [ ]:
print(len(countries_kept))

We see that there are 119 countries left, including the top 10 countries by GDP.

If we only keep the covariates mentioned above and we only keep countries for which all covariates are recorded, what do the correlations between covariates look like? What about the correlations between the covariates and total deaths per million by the end of the pandemic?

In [ ]:
#Covariates that we decided not to drop
newcols = ['population_density', 'median_age', 'gdp_per_capita', 'extreme_poverty', 'hospital_beds_per_thousand', 'life_expectancy']
df_vars = df.drop_duplicates(subset = 'country', keep = 'last')[['country'] + ['total_deaths_per_million'] + newcols]
df_vars = df_vars[df_vars['country'].isin(countries_kept)]

In [ ]:
sns.heatmap(df_vars[['total_deaths_per_million'] + newcols].corr(), vmin = -1, vmax = 1, annot = True, cmap = 'coolwarm')
pass

In [ ]:
sns.pairplot(df_vars)
pass

We see that total deaths per million is positively correlated with median age, which makes sense since the elderly are more vulnerable to Covid. Surprisingly, total deaths per million has positive correlation with covariates like life expectancy and hospital beds per thousand. This may be explained by a confounding variable: high life expectancy and hospital beds per thousand is also associated with high median age.

We also see that there are some extreme outliers in covariates such as population density and life expectancy. What are the top three and bottom three countries for each variable?

In [ ]:
for col in ['total_deaths_per_million'] + newcols:
    print(f'Maximum {col}')
    display(df_vars[df_vars[col].isin(df_vars[col].nlargest(3))])
    print(f'Minimum {col}')
    display(df_vars[df_vars[col].isin(df_vars[col].nsmallest(3))])

What are the summary statistics for each variable?

In [ ]:
df_vars[['total_deaths_per_million'] + newcols].describe()

Many of the marginal distributions are heavily right-tailed. It might make more sense to measure these variables on the log scale.

In [ ]:
for col in ['total_deaths_per_million', 'population_density', 'gdp_per_capita', 'extreme_poverty', 'hospital_beds_per_thousand']:
    df_vars[col] = np.log(df_vars[col])
    df_vars = df_vars.rename(columns={col: 'log_' + col})

New correlation matrix and pairplot:

In [ ]:
sns.heatmap(df_vars.drop(columns = ['country']).corr(), vmin = -1, vmax = 1, annot = True, cmap = 'coolwarm')
pass

In [ ]:
sns.pairplot(df_vars)
pass

We see that several of the marginal distributions are approximately Normal when the variable is measured on the log scale. Thus, it seems reasonable to assume that these variables follow a Log Normal distribution. We also see strong linear relationships between the covariates and total deaths per million, and between different covariates, when the variables are measured on the log scale.